## 0. Imports/Constants

In [73]:
import uproot
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys,os
import argparse
from tqdm import tqdm
import glob

#CAFPYANA working directory
CAFPYANA_WD = '/exp/sbnd/app/users/brindenc/develop/cafpyana'
os.environ['CAFPYANA_WD'] = CAFPYANA_WD

cafpyana_wd = os.environ.get('CAFPYANA_WD')
if cafpyana_wd and cafpyana_wd not in sys.path:
    sys.path.insert(0, cafpyana_wd)
    sys.path.insert(0, cafpyana_wd + '/pyanalib')

#My imports 
SBNDANA_DIR = '/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd'
sys.path.insert(0,SBNDANA_DIR)
sys.path.insert(0,f'{SBNDANA_DIR.replace("/numuincl/sbnd","/numuincl")}')
plt.style.use(f'{SBNDANA_DIR}/plotlibrary/numu2025.mplstyle')

from sbnd.cafclasses.slice import CAFSlice
from sbnd.cafclasses.pfp import PFP
from sbnd.cafclasses.nu import NU
from sbnd.constants import *
from sbnd.numu.numu_constants import *
from sbnd.detector.definitions import *
from sbnd.general import plotters
from naming import *
from sbnd.general.utils import read_hdf

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [142]:
# Fill these
checkpoint = 'checkpoint7_denttest'
VERSION = 'v8'
NCPU = 16
SMALL = True

PLOT_DIR = f'{DATA_DIR}/plots/{checkpoint}/dent_studies'

#MC Fnames
MC_FNAMES = glob.glob(
  f'{DATA_DIR}/mc_syst/{VERSION}/*_nosyst*/*.df'
)
DATA_OFFBEAM_FNAMES = glob.glob(
  f'{DATA_DIR}/offbeam/{VERSION}/*data*/*.df'
)
DATA_FNAMES = glob.glob(
  f'{DATA_DIR}/data/{VERSION}/*dataonbeam*/*.df'
)
MC_LOWE_FNAMES = glob.glob(
  f'{DATA_DIR}/mc_lowe/{VERSION}/*_mclowe_*/*.df'
)

if SMALL:
  MC_FNAMES = MC_FNAMES[:len(MC_FNAMES)//10]
  DATA_OFFBEAM_FNAMES = DATA_OFFBEAM_FNAMES[:len(DATA_OFFBEAM_FNAMES)//10]
  MC_LOWE_FNAMES = MC_LOWE_FNAMES[:len(MC_LOWE_FNAMES)//10]
else:
  MC_FNAMES = MC_FNAMES[:len(MC_FNAMES)//3]
  DATA_OFFBEAM_FNAMES = DATA_OFFBEAM_FNAMES[:len(DATA_OFFBEAM_FNAMES)//3]
  MC_LOWE_FNAMES = MC_LOWE_FNAMES[:len(MC_LOWE_FNAMES)//3]

## 1. Load 

### 1.1 Get Exposures

In [143]:
POT_LOWE = read_hdf(MC_LOWE_FNAMES,key=POT_KEY, ncpu=NCPU, show_progress=True).TotalPOT.sum()
print(f'POT_LOWE: {POT_LOWE:.2e}')
POT_MC = read_hdf(MC_FNAMES,key=POT_KEY, ncpu=NCPU, show_progress=True).TotalPOT.sum()
print(f'POT_MC: {POT_MC:.2e}')
LIVETIME_OFFBEAM = read_hdf(DATA_OFFBEAM_FNAMES,key=HDR_KEY, ncpu=NCPU, show_progress=True).noffbeambnb.values.sum()
print(f'LIVETIME_OFFBEAM: {LIVETIME_OFFBEAM:.2e}')
POT_DATA = read_hdf(DATA_FNAMES,key=POT_KEY, ncpu=NCPU, show_progress=True).TotalPOT.sum()
LIVETIME_DATA = 9.51e5 # Temporary override
print(f'LIVETIME_DATA: {LIVETIME_DATA:.2e}')
POT_LABEL = f'{POT_DATA:.2e} POT'
print(f'POT_LABEL: {POT_LABEL}')

read_hdf:   0%|          | 0/50 [00:00<?, ?it/s]

read_hdf: 100%|██████████| 50/50 [00:03<00:00, 13.56it/s]

POT_LOWE: 1.31e+19



read_hdf: 100%|██████████| 100/100 [00:06<00:00, 15.44it/s]


POT_MC: 5.62e+19


read_hdf: 100%|██████████| 50/50 [00:02<00:00, 17.11it/s]


LIVETIME_OFFBEAM: 1.74e+06


read_hdf: 100%|██████████| 10/10 [00:01<00:00,  5.57it/s]


LIVETIME_DATA: 9.51e+05
POT_LABEL: 5.95e+18 POT


### 1.2 Get Slices

In [144]:
# Get slices (apply cosmic cuts)
slc_mc = CAFSlice.load(MC_FNAMES,
  key=PAND_KEY,
  cuts=PAND_CUTS_BASE[:-1], #All but muon cut
  ncpu=NCPU,
  show_progress=True,
  verbose=False
)
slc_lowe = CAFSlice.load(MC_LOWE_FNAMES,
  key=PAND_KEY,
  cuts=PAND_CUTS_BASE[:-1], # All but muon cut
  ncpu=NCPU,
  show_progress=True,
  verbose=False
)
slc_offbeam = CAFSlice.load(DATA_OFFBEAM_FNAMES,
  key=PAND_KEY,
  cuts=PAND_CUTS_BASE[:-1], # All but muon cut
  ncpu=NCPU,
  show_progress=True,
  verbose=False)
slc_data = CAFSlice.load(DATA_FNAMES,
  key=PAND_KEY,
  cuts=PAND_CUTS_BASE[:-1], # All but muon cut
  ncpu=NCPU,
  show_progress=True,
  verbose=False
)

In [145]:
for s in [slc_mc,slc_lowe,slc_offbeam,slc_data]:
  s.cut_is_cont(cut=False) # Add cont_full cut
  s.apply_cut('cut.cont_full',cut=True) # Consider only contained muons

Applied cut on key: cut.cont_full (247,934 --> 82,310)
Applied cut on key: cut.cont_full (281 --> 13)
Applied cut on key: cut.cont_full (2,035 --> 93)
Applied cut on key: cut.cont_full (32,122 --> 9,782)


### 1.3 Get PFPs

In [146]:
# Get pfps
pfp_mc = PFP.load(MC_FNAMES,
  key=PFP_KEY,
  ncpu=NCPU,
  show_progress=True,
  verbose=True
)
pfp_lowe = PFP.load(MC_LOWE_FNAMES,
  key=PFP_KEY,
  ncpu=NCPU,
  show_progress=True,
  verbose=True
)
pfp_offbeam = PFP.load(DATA_OFFBEAM_FNAMES,
  key=PFP_KEY,
  ncpu=NCPU,
  show_progress=True,
  verbose=True
)
pfp_data = PFP.load(DATA_FNAMES,
  key=PFP_KEY,
  ncpu=NCPU,
  show_progress=True,
  verbose=True
)

Loading combined CAF: 100%|██████████| 10/10 [00:03<00:00,  2.94it/s]


In [147]:
def filter_pfp(pfp,slc):
  pfp.data = slc.get_reference_df(pfp)
  m = pfp.data.pfp.trk.len > 32 # 32 cm cut
  pfp.data = pfp.data[m]
  return pfp

# Filter PFPs down to slice
pfp_mc = filter_pfp(pfp_mc,slc_mc)
pfp_lowe = filter_pfp(pfp_lowe,slc_lowe)
pfp_offbeam = filter_pfp(pfp_offbeam,slc_offbeam)
pfp_data = filter_pfp(pfp_data,slc_data)

### 1.4 Scale and combine

In [148]:
# Scale
slc_mc.scale_to_pot(POT_DATA,sample_pot=POT_MC,overwrite=True)
slc_lowe.scale_to_pot(POT_DATA,sample_pot=POT_LOWE,overwrite=True)
slc_offbeam.scale_to_livetime(LIVETIME_DATA,sample_livetime=LIVETIME_OFFBEAM,overwrite=True)

pfp_mc.scale_to_pot(POT_DATA,sample_pot=POT_MC,overwrite=True)
pfp_lowe.scale_to_pot(POT_DATA,sample_pot=POT_LOWE,overwrite=True)
pfp_offbeam.scale_to_livetime(LIVETIME_DATA,sample_livetime=LIVETIME_OFFBEAM,overwrite=True)

--scaling to POT (1.06e-01): 5.62e+19 -> 5.95e+18
--scaling to POT (4.54e-01): 1.31e+19 -> 5.95e+18
--scaling to livetime (5.45e-01): 1.74e+06 --> 9.51e+05
--scaling to POT (1.06e-01): 5.62e+19 -> 5.95e+18
--scaling to POT (4.54e-01): 1.31e+19 -> 5.95e+18
--scaling to livetime (5.45e-01): 1.74e+06 --> 9.51e+05


In [149]:
# Combine
slc_mc.combine(slc_offbeam,duplicate_ok=True)
#del slc_offbeam
slc_mc.combine(slc_lowe,duplicate_ok=True)
#del slc_lowe

pfp_mc.combine(pfp_offbeam,duplicate_ok=True)
#del pfp_offbeam
pfp_mc.combine(pfp_lowe,duplicate_ok=True)
#del pfp_lowe


## 2. Slice variables

### 2.1 Break into list of objects

In [150]:
slcs = []

#Get event type col
slc_event_type_col = slc_mc.get_key('truth.event_type')

# Group data by event type once
slc_groups = slc_mc.data.groupby(slc_event_type_col)

#Create objects only for groups that exist
for key,val in EVENT_TYPE_LIST.items():
  if key in slc_groups.groups:
    slcs.append(CAFSlice(slc_groups.get_group(key)))

/tmp/ipykernel_1851/3773502384.py:12: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  slcs.append(CAFSlice(slc_groups.get_group(key)))


In [151]:
pfps = []

# Get event type col
pfp_mc.add_cols('pfp.trk.truth.p.abspdg',np.abs(pfp_mc.data.pfp.trk.truth.p.pdg.values),fill=np.int32(0))
pfp_true_particle = pfp_mc.get_key('pfp.trk.truth.p.abspdg')

# Group by true particle
pfp_groups = pfp_mc.data.groupby(pfp_true_particle)
missing_keys = []

# Create objects only for groups that exist
for key,val in PARTICLE_TYPE_LIST.items():
  if key in pfp_groups.groups:
    pfps.append(PFP(pfp_groups.get_group(key)))
  else:
    missing_keys.append(key)


/tmp/ipykernel_1851/4185893548.py:14: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  pfps.append(PFP(pfp_groups.get_group(key)))


In [152]:
#Get label info
labels = []
names = []
colors = []

for key,item in EVENT_TYPE_LIST.items():
  if key == -1:# or key == 4: #Skip nuecc since it's not in small sample
    continue
  labels.append(item[0])
  names.append(item[1])
  colors.append(item[2])

slc_weights = [s.data.genweight for s in slcs]
assert [any(np.isnan(sw)) or any(np.isinf(sw)) for sw in slc_weights].count(True) == 0

### 2.2 Make plots

In [153]:
bcfm_bins = np.arange(0, 1.1, 0.025)
flashpe_bins = np.arange(0, 10500, 500)
xy_bins = np.arange(-210, 220, 10)
z_bins = np.arange(-10, 520, 10)
len_bins = np.arange(0, 750, 25)
dphi = np.pi / 12
phi_bins = np.arange(-np.pi, np.pi + dphi, dphi)
theta_bins = np.arange(0, np.pi + dphi, dphi)

In [154]:
from sbnd.detector.volume import involume
from sbnd.plotlibrary import makeplot
cuts = ['precut','cut.muon','cut.cont']
folders = ['before_muon_cut','cont_full','cont_tpc']

for i, cut in enumerate(cuts):
  print('*' * 70)
  print(f'cut: {cut}')
  
  # Only copy when we need to apply cuts
  if i == 1:  # First cut application
    _slcs = [s.copy() for s in slcs]
    _slc_data = slc_data.copy()
    _pfps = [p.copy() for p in pfps]
    pfp_data = pfp_data.copy()
    print('copied')
  else:
    _slcs = slcs #no cuts
    _slc_data = slc_data #no cuts
  if cut != 'precut':
    for s in _slcs:
      s.apply_cut(cut)

    _slc_data.apply_cut(cut)
  
  _data_events = len(_slc_data.data)
  print(f'len(_slc_data): {_data_events}')
  
  # Divide slcs into octants based on vertex
  _octant_slcs = {q: [] for q in range(len(OCTANTS))}
  for j, octant in enumerate(OCTANTS):
      for s in _slcs:
          m = involume(s.data.slc.vertex, octant)
          _octant_slcs[j].append(CAFSlice(s.data[m]))
  _octant_slc_data = []
  if _slc_data is not None:
      for octant in OCTANTS:
          m = involume(_slc_data.data.slc.vertex, octant)
          _octant_slc_data.append(CAFSlice(_slc_data.data[m]))
  
  
  for j,(olabel,oname,_oslc_data) in tqdm(enumerate(zip(OCTANT_LABELS,OCTANT_NAMES,_octant_slc_data)),total=len(OCTANT_LABELS),desc='Making plots'):
    _oslcs = _octant_slcs[j]
    plot_dir = f'{PLOT_DIR}/{folders[i]}/{olabel}'
    # MC
    _weights = [s.data.genweight for s in _oslcs]
    # - Muon information
    costheta = [s.data.mu.pfp.trk.costheta for s in _oslcs]
    momentum = [s.data.mu.pfp.trk.P.p_muon for s in _oslcs]
    bin2d = [s.data.bin.differential for s in _oslcs]
    length = [s.data.mu.pfp.trk.len for s in _oslcs]

    # -start and end points
    start_x = [s.data.mu.pfp.trk.start.x for s in _oslcs]
    start_y = [s.data.mu.pfp.trk.start.y for s in _oslcs]
    start_z = [s.data.mu.pfp.trk.start.z for s in _oslcs]
    end_x = [s.data.mu.pfp.trk.end.x for s in _oslcs]
    end_y = [s.data.mu.pfp.trk.end.y for s in _oslcs]
    end_z = [s.data.mu.pfp.trk.end.z for s in _oslcs]

    # -phi and theta
    phi = [s.data.mu.pfp.trk.phi for s in _oslcs]
    theta = [s.data.mu.pfp.trk.theta for s in _oslcs]

    # -interaction vertex
    interaction_x = [s.data.slc.vertex.x for s in _oslcs]
    interaction_y = [s.data.slc.vertex.y for s in _oslcs]
    interaction_z = [s.data.slc.vertex.z for s in _oslcs]

    # Data
    # - Muon information
    costheta_data = _oslc_data.data.mu.pfp.trk.costheta
    momentum_data = _oslc_data.data.mu.pfp.trk.P.p_muon
    bin2d_data = _oslc_data.data.bin.differential
    length_data = _oslc_data.data.mu.pfp.trk.len

    # -start and end points
    start_x_data = _oslc_data.data.mu.pfp.trk.start.x
    start_y_data = _oslc_data.data.mu.pfp.trk.start.y
    start_z_data = _oslc_data.data.mu.pfp.trk.start.z
    end_x_data = _oslc_data.data.mu.pfp.trk.end.x
    end_y_data = _oslc_data.data.mu.pfp.trk.end.y
    end_z_data = _oslc_data.data.mu.pfp.trk.end.z

    # -interaction vertex
    interaction_x_data = _oslc_data.data.slc.vertex.x
    interaction_y_data = _oslc_data.data.slc.vertex.y
    interaction_z_data = _oslc_data.data.slc.vertex.z

    # -phi and theta
    phi_data = _oslc_data.data.mu.pfp.trk.phi
    theta_data = _oslc_data.data.mu.pfp.trk.theta

    # -interaction vertex
    interaction_x_data = _oslc_data.data.slc.vertex.x
    interaction_y_data = _oslc_data.data.slc.vertex.y
    interaction_z_data = _oslc_data.data.slc.vertex.z

    #Muon infomation
    #-Costheta
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        costheta, labels, False,
        data_series=costheta_data,
        #cut_desc=cut_desc,
        xlabel=r'$\cos\theta_{\mu}$',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=np.arange(-1, 1.1, 0.1),
        cut=cut,
        savename=f'costhetamu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    #-Momentum
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        momentum, labels, False,
        data_series=momentum_data,
        xlabel=r'$p_{\mu}$ [GeV]',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=np.arange(0, 2.1, 0.1),
        cut=cut,
        savename=f'momentummu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    #length
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        length, labels, False,
        data_series=length_data,
        xlabel=r'$L_{\mu}$ [cm]',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=len_bins,
        cut=cut,
        savename=f'lengthmu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    # -phi
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        phi, labels, False,
        data_series=phi_data,
        xlabel=r'$\phi_{\mu}$ [rad]',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=phi_bins,
        cut=cut,
        savename=f'phimu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    # -theta
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        theta, labels, False,
        data_series=theta_data,
        xlabel=r'$\theta_{\mu}$ [rad]',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=theta_bins,
        cut=cut,
        savename=f'thetamu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    # start x
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        start_x, labels, False,
        data_series=start_x_data,
        xlabel=r'Reconstructed $\mu$ start x',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=xy_bins,
        cut=cut,
        savename=f'startxmu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    # start y
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        start_y, labels, False,
        data_series=start_y_data,
        xlabel=r'Reconstructed $\mu$ start y',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=xy_bins,
        cut=cut,
        savename=f'startymu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    # start z
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        start_z, labels, False,
        data_series=start_z_data,
        xlabel=r'Reconstructed $\mu$ start z',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=z_bins,
        cut=cut,
        savename=f'startzmu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    # end x
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        end_x, labels, False,
        data_series=end_x_data,
        xlabel=r'Reconstructed $\mu$ end x',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=xy_bins,
        cut=cut,
        savename=f'endxmu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    # end y
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        end_y, labels, False,
        data_series=end_y_data,
        xlabel=r'Reconstructed $\mu$ end y',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=xy_bins,
        cut=cut,
        savename=f'endymu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    # end z
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        end_z, labels, False,
        data_series=end_z_data,
        xlabel=r'Reconstructed $\mu$ end z',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=z_bins,
        cut=cut,
        savename=f'endzmu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    # interaction vertex
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        interaction_x, labels, False,
        data_series=interaction_x_data,
        xlabel=r'Interaction x [cm]',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=xy_bins,
        cut=cut,
        savename=f'vtxxmu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    # interaction y
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        interaction_y, labels, False,
        data_series=interaction_y_data,
        xlabel=r'Interaction y [cm]',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=xy_bins,
        cut=cut,
        savename=f'vtymu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

    # interaction z
    fig, ax, ax2 = makeplot.create_hist_dataratio(
        interaction_z, labels, False,
        data_series=interaction_z_data,
        xlabel=r'Interaction z [cm]',
        label=INTERNAL_DATA_LABEL,
        colors=colors,
        weights=_weights,
        bins=z_bins,
        cut=cut,
        savename=f'vtzmu_{cut}cut_octant{i}',
        plot_dir=plot_dir,
        data_events=_data_events,
        pot_label=POT_LABEL,
        scale_data=False,
        show_counts=True,
        title=oname
    )

**********************************************************************
cut: precut
len(_slc_data): 9782


Making plots:   0%|          | 0/8 [00:00<?, ?it/s]/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plotlibrary/makeplot.py:122: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data_counts = data_series.groupby(pd.cut(data_series,bins=bins)).count()
/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plotlibrary/makeplot.py:598: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data_counts = np.array(data_series.groupby(pd.cut(data_series,bins=bins)).count().values,dtype=np.float32)
/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plot

**********************************************************************
cut: cut.muon
copied
Applied cut on key: cut.muon (54,719 --> 54,719)
Applied cut on key: cut.muon (20,009 --> 20,009)
Applied cut on key: cut.muon (1,881 --> 1,881)
Applied cut on key: cut.muon (1,351 --> 1,351)
Applied cut on key: cut.muon (100 --> 100)
Applied cut on key: cut.muon (3,542 --> 3,542)
Applied cut on key: cut.muon (814 --> 814)
Applied cut on key: cut.muon (9,782 --> 9,782)
len(_slc_data): 9782


Making plots:   0%|          | 0/8 [00:00<?, ?it/s]/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plotlibrary/makeplot.py:122: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data_counts = data_series.groupby(pd.cut(data_series,bins=bins)).count()
/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plotlibrary/makeplot.py:598: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data_counts = np.array(data_series.groupby(pd.cut(data_series,bins=bins)).count().values,dtype=np.float32)
/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plot

**********************************************************************
cut: cut.cont
Applied cut on key: cut.cont (54,719 --> 54,606)
Applied cut on key: cut.cont (20,009 --> 837)
Applied cut on key: cut.cont (1,881 --> 1,658)
Applied cut on key: cut.cont (1,351 --> 1,162)
Applied cut on key: cut.cont (100 --> 88)
Applied cut on key: cut.cont (3,542 --> 3,160)
Applied cut on key: cut.cont (814 --> 710)
Applied cut on key: cut.cont (9,782 --> 7,724)
len(_slc_data): 7724


Making plots:   0%|          | 0/8 [00:00<?, ?it/s]/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plotlibrary/makeplot.py:122: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data_counts = data_series.groupby(pd.cut(data_series,bins=bins)).count()
/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plotlibrary/makeplot.py:598: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data_counts = np.array(data_series.groupby(pd.cut(data_series,bins=bins)).count().values,dtype=np.float32)
/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plot

## 3. PFP plots

Goal - Find chi2 values in each TPC before and after containment and tpc containment cut.

### 3.1 Create particle groups

In [155]:
# (      'pfp',               'trk', 'is_contained', ...),
# (      'pfp',               'trk',    'cont_tpc0', ...),
# (      'pfp',               'trk',    'cont_tpc1', ...)

In [156]:
pfps = []

# Get event type col
pfp_mc.add_cols('pfp.trk.truth.p.abspdg',np.abs(pfp_mc.data.pfp.trk.truth.p.pdg.values),fill=np.int32(0))
pfp_true_particle = pfp_mc.get_key('pfp.trk.truth.p.abspdg')

# Group by true particle
pfp_groups = pfp_mc.data.groupby(pfp_true_particle)
missing_keys = []

# Create objects only for groups that exist
for key,val in PARTICLE_TYPE_LIST.items():
  if key in pfp_groups.groups:
    pfps.append(PFP(pfp_groups.get_group(key)))
  else:
    missing_keys.append(key)

/tmp/ipykernel_1851/3852641235.py:14: FutureWarning: When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.
  pfps.append(PFP(pfp_groups.get_group(key)))


In [157]:
#Get pfp stuff
pfp_labels = []
pfp_names = []
pfp_colors = []

for key,item in PARTICLE_TYPE_LIST.items():
  if key in missing_keys:
    continue
  pfp_labels.append(item[0])
  pfp_names.append(item[1])
  pfp_colors.append(item[2])

### 3.2 Create plots

In [158]:
track_score_bins = np.arange(0, 1.025, 0.025)
chi2mu_bins = np.arange(0,50,2)
chi2p_bins = np.arange(0,300,5)

In [159]:
from sbnd.detector.volume import involume
from sbnd.plotlibrary import makeplot
folders = ['cont_full','cont_tpc']

for i, folder in enumerate(folders):
  print('*' * 70)
  print(f'folder: {folder}')
  
  # Only copy when we need to apply cuts
  if i == 0:  # First cut application
    _pfps = [p.copy() for p in pfps]
    _pfp_data = pfp_data.copy()
    print('copied')
  else:
    _pfps = _pfps #no cuts
    _pfp_data = _pfp_data #no cuts

  # Apply the cut
  for p in _pfps:
    if folder == 'cont_full':
      m = p.data.pfp.trk.is_contained
      p = PFP(p.data[m])
      print(f'-- {PARTICLE_TYPE_LIST[p.data.pfp.trk.truth.p.abspdg.values[0]][1]} {len(m)} -> {m.sum()}')
    elif folder == 'cont_tpc':
      m = p.data.pfp.trk.cont_tpc0 | p.data.pfp.trk.cont_tpc1
      p = PFP(p.data[m])
      print(f'-- {len(m)} -> {m.sum()}')
  if folder == 'cont_full':
    m = _pfp_data.data.pfp.trk.is_contained
    _pfp_data = PFP(_pfp_data.data[m])
    print(f'- {len(_pfp_data.data)} -> {len(_pfp_data.data)}')
  elif folder == 'cont_tpc':
    m = _pfp_data.data.pfp.trk.cont_tpc0 | _pfp_data.data.pfp.trk.cont_tpc1
    _pfp_data = PFP(_pfp_data.data[m])
    print(f'- {len(_pfp_data.data)} -> {len(_pfp_data.data)}')


  
  _data_events = len(_pfp_data.data)
  print(f'len(_pfp_data): {_data_events}')
  
  # Divide pfps into octants based on vertex
  _octant_pfps = {q: [] for q in range(len(OCTANTS))}
  for j, octant in enumerate(OCTANTS):
      for p in _pfps:
          m = involume(p.data.pfp.trk.start, octant)
          _octant_pfps[j].append(PFP(p.data[m]))
  _octant_pfp_data = []
  if _pfp_data is not None:
      for octant in OCTANTS:
          m = involume(_pfp_data.data.pfp.trk.start, octant)
          _octant_pfp_data.append(PFP(_pfp_data.data[m]))
  
  for j,(olabel,oname,_opfp_data) in tqdm(enumerate(zip(OCTANT_LABELS,OCTANT_NAMES,_octant_pfp_data)),total=len(OCTANT_LABELS),desc='Making plots'):
    _opfps = _octant_pfps[j]
    plot_dir = f'{PLOT_DIR}/{folders[i]}/{olabel}'

    # Extract chi2 values
    chi2_muons = [p.data.pfp.trk.chi2pid.I2.chi2_muon for p in _opfps]
    chi2_protons = [p.data.pfp.trk.chi2pid.I2.chi2_proton for p in _opfps]
    weights = [p.data.genweight for p in _opfps]

    chi2_muons_data = _opfp_data.data.pfp.trk.chi2pid.I2.chi2_muon
    chi2_protons_data = _opfp_data.data.pfp.trk.chi2pid.I2.chi2_proton

    #chi2mu
    fig,ax,ax2 = makeplot.create_hist_dataratio(
      chi2_muons,
      pfp_labels,
      False,
      bins=chi2mu_bins,
      data_series=chi2_muons_data,
      weights=weights,
      #cut_desc='',
      xlabel=r'$\chi^2_{\mu}$',
      colors=pfp_colors,
      label=INTERNAL_DATA_LABEL,
      #savename=f'chi2mu_{olabel}',
      plot_dir=plot_dir,
      data_events=_data_events,
      pot_label=POT_LABEL,
      show_counts=False,
      title=oname
    )
    #chi2p
    fig,ax,ax2 = makeplot.create_hist_dataratio(
      chi2_protons,
      pfp_labels,
      False,
      bins=chi2p_bins,
      data_series=chi2_protons_data,
      weights=weights,
      #cut_desc='',
      xlabel=r'$\chi^2_{p}$',
      colors=pfp_colors,
      label=INTERNAL_DATA_LABEL,
      savename=f'chi2p_{olabel}',
      plot_dir=plot_dir,
      data_events=_data_events,
      pot_label=POT_LABEL,
      show_counts=False,
      title=oname
    )
    #break
  #break


**********************************************************************
folder: cont_full
copied
-- electron 50 -> 26
-- photon 387 -> 363
-- muon 80479 -> 74673
-- pi 13243 -> 12367
-- proton 22583 -> 20841
-- kaon 100 -> 100
- 12262 -> 12262
len(_pfp_data): 12262


Making plots:   0%|          | 0/8 [00:00<?, ?it/s]/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plotlibrary/makeplot.py:122: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data_counts = data_series.groupby(pd.cut(data_series,bins=bins)).count()
/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plotlibrary/makeplot.py:598: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data_counts = np.array(data_series.groupby(pd.cut(data_series,bins=bins)).count().values,dtype=np.float32)
/exp/sbnd/app/users/brindenc/develop/cafpyana/analysis_village/numuincl/sbnd/plot

**********************************************************************
folder: cont_tpc
-- 50 -> 26
-- 387 -> 287
-- 80479 -> 54945
-- 13243 -> 10962
-- 22583 -> 19084
-- 100 -> 75
- 9895 -> 9895
len(_pfp_data): 9895


Making plots: 100%|██████████| 8/8 [00:13<00:00,  1.70s/it]


In [160]:
plot_dir

'/exp/sbnd/data/users/brindenc/analyze_sbnd/numu/v10_06_00_validation/pandora/plots/checkpoint7_denttest/dent_studies/cont_tpc/bottom_left_high_z'